# 합성 RGB→T1 Colab GPU 통합 pilot

테스트 세션이 고정한 ready notebook과 **source ZIP 한 개**만 사용합니다. 샘플 ZIP·Drive·다운로드·자동 설치·실데이터 학습은 없습니다. GPU FP32/AMP 각각 RGB와3D의 full4 대 prefix1+resume3을 비교합니다. native1024×1024/합성62관절/train256×256/B2는 계산량 profile이며 실제 관절 의미/정확도가 아닙니다.

한 번 실행한 job은 다시 학습하지 않습니다. 중단/실패/자원 초과이면 예산이나 크기를 바꾸지 말고 마지막 셀로 ZIP을 회수하세요. 실제 GPU는 이 notebook을 테스트 세션에서 실행하기 전까지 미검증입니다. Drive 새VM과 실제 release 근거는 이 합성 시험으로 해결되지 않습니다. source 업로드64MiB/해제128MiB 공간은 결과256MiB 예산 밖입니다.


In [ ]:
# 고정 GPU pilot source ZIP만 업로드. 샘플 ZIP/자동 설치/Drive 연결 없음.
import sys, os, io, json, stat, re, hashlib, tempfile, zipfile
from pathlib import Path, PurePosixPath
from google.colab import files

EXPECTED_REVISION = ""
EXPECTED_SOURCE_SHA256 = ""
JOB_ID = ""
if not re.fullmatch(r"[0-9a-f]{40}", EXPECTED_REVISION) or not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_SOURCE_SHA256):
    raise RuntimeError("테스트 세션의 고정 revision/source ZIP SHA256을 설정한 ready notebook이 필요합니다.")
if not re.fullmatch(r"[A-Za-z0-9_-]{1,64}", JOB_ID):
    raise RuntimeError("테스트 세션이 고정한 JOB_ID가 필요합니다.")
if "PILOT_SESSION_ROOT" in globals():
    raise RuntimeError("이미 준비한 실행입니다. 기존 결과를 회수하고 중단하세요.")
PILOT_SESSION_ROOT = Path("/content")/("rewind_gpu_pilot_"+JOB_ID)
PILOT_SESSION_ROOT.mkdir(exist_ok=False)
SOURCE_OK = False
receipt = {"status":"failed", "expected_revision":EXPECTED_REVISION, "expected_zip_sha256":EXPECTED_SOURCE_SHA256,
           "identity_scope":"ZIP bytes + packaged source stamp comparison; not Git history authentication"}
try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("고정 source ZIP 한 개만 필요합니다.")
    source_name, raw = next(iter(uploaded.items()))
    if not source_name.lower().endswith(".zip") or len(raw) > 64*1024**2:
        raise ValueError("source ZIP 64MiB 상한")
    actual = hashlib.sha256(raw).hexdigest()
    receipt.update(name=source_name, bytes=len(raw), sha256=actual)
    if actual != EXPECTED_SOURCE_SHA256:
        raise ValueError("source ZIP SHA256 불일치")
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        members = archive.infolist()
        if len(members)>5000 or sum(m.file_size for m in members)>128*1024**2:
            raise ValueError("소스 해제 상한 초과")
        roots = [PurePosixPath(m.filename).parent.parent for m in members if m.filename.endswith("rewind_m1/gpu_pilot.py")]
        if len(roots)!=1:
            raise ValueError("통합 worker가 있는 단일 source ZIP 필요")
        seen = set()
        excluded = {".git",".venv","data","outputs","checkpoints",".local-archive",".aws",".codex",".agents","__pycache__"}
        for m in members:
            p = PurePosixPath(m.filename)
            mode = stat.S_IFMT(m.external_attr>>16)
            if (p.is_absolute() or ".." in p.parts or "\\" in m.filename or not p.parts
                or any(":" in x or x.endswith(("."," ")) for x in p.parts)
                or any(ord(c)<32 for c in m.orig_filename)
                or mode not in (0,stat.S_IFREG,stat.S_IFDIR) or m.flag_bits&1):
                raise ValueError("안전하지 않은 source ZIP")
            key = p.as_posix().casefold().rstrip("/")
            if key in seen:
                raise ValueError("중복 source 경로")
            seen.add(key)
            if any(x in excluded or x==".env" or x.startswith(".env.") for x in p.parts):
                raise ValueError("소스 ZIP에 제외 대상 파일이 있습니다.")
        extract_root = PILOT_SESSION_ROOT/"source"
        extract_root.mkdir()
        archive.extractall(extract_root)
    PROJECT_ROOT = (extract_root/roots[0]).resolve()
    if not PROJECT_ROOT.is_relative_to(extract_root.resolve()):
        raise ValueError("source root 이탈")
    stamp = (PROJECT_ROOT/"SOURCE_REVISION.txt").read_text(encoding="utf-8").strip()
    if stamp != EXPECTED_REVISION:
        raise ValueError("SOURCE_REVISION stamp 불일치")
    receipt["packaged_code_sha256"] = {p.relative_to(PROJECT_ROOT).as_posix():hashlib.sha256(p.read_bytes()).hexdigest()
        for package in ("rewind_m1","rewind_data") for p in sorted((PROJECT_ROOT/package).glob("*.py"))}
    SOURCE_OK = True
    receipt.update(status="ready", source_stamp=stamp)
except Exception as exc:
    receipt["error_type"] = type(exc).__name__
finally:
    (PILOT_SESSION_ROOT/"source_upload.json").write_text(json.dumps(receipt,ensure_ascii=False),encoding="utf-8")
    for name in ("uploaded","raw"):
        globals().pop(name,None)
print(receipt)


In [ ]:
# 기존 Colab GPU 환경과 여유 자원 검사. 설치나 CPU fallback을 하지 않는다.
import importlib.util, importlib.metadata, platform, shutil

environment = {"python":platform.python_version(), "source_ready":SOURCE_OK, "profile":"gpu"}
for package,module in (("torch","torch"),("Pillow","PIL"),("psutil","psutil")):
    try: version=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: version=None
    environment[package]={"present":importlib.util.find_spec(module) is not None,"version":version}
ENV_READY=False
try:
    ENV_READY=SOURCE_OK and all(environment[p]["present"] for p in ("torch","Pillow","psutil"))
    if ENV_READY:
        sys.path.insert(0,str(PROJECT_ROOT))
        import torch, psutil
        match=re.match(r"^(\d+)\.(\d+)",str(torch.__version__))
        cuda=torch.cuda.is_available()
        environment.update(cuda_available=cuda,torch_cuda_runtime=torch.version.cuda,
            host_total_bytes=psutil.virtual_memory().total,host_available_bytes=psutil.virtual_memory().available,
            disk_free_bytes=shutil.disk_usage(PILOT_SESSION_ROOT).free)
        ENV_READY=bool(match and tuple(map(int,match.groups())) >= (2,3) and cuda and
                       environment["host_available_bytes"]>=4*1024**3 and environment["disk_free_bytes"]>=256*1024**2)
        if cuda:
            props=torch.cuda.get_device_properties(0); free,total=torch.cuda.mem_get_info()
            environment.update(gpu_name=props.name,gpu_total_bytes=props.total_memory,gpu_free_bytes=free)
            ENV_READY=ENV_READY and free>=2*1024**3
except BaseException as exc:
    ENV_READY=False
    environment["error"]={"type":type(exc).__name__,"reason":str(exc)[:256]}
finally:
    environment["ready"]=ENV_READY
    (PILOT_SESSION_ROOT/"environment.json").write_text(json.dumps(environment,ensure_ascii=False,indent=2),encoding="utf8")
print(environment)
if not ENV_READY:
    print("중단: GPU/의존성/자원 검사 실패. 설치·자동축소·CPU fallback 없이 회수 셀로 이동하세요.")


In [ ]:
# 한 번만 실행. timeout/사용자 중단에도 child를 종료하고 디스크 결과를 보존한다.
import subprocess, threading, time
if globals().get("PILOT_STARTED",False) or (PILOT_SESSION_ROOT/"supervision_started.json").exists():
    raise RuntimeError("이 job은 이미 소비됐습니다. 새 학습 없이 결과만 회수하세요.")
PILOT_STARTED=True
with (PILOT_SESSION_ROOT/"supervision_started.json").open("x",encoding="utf8") as f:
    json.dump({"job_id":JOB_ID,"profile":"gpu","worker_pid":None},f)
supervision={"status":"failed","timeout_seconds":600,"started":False,"profile":"gpu"}
process=None; threads=[]; overflow=threading.Event(); pump_errors=[]; cleanup_errors=[]
started=time.monotonic()
def pump(pipe,path):
    size=0
    try:
        with path.open("xb") as log:
            while True:
                chunk=pipe.read(4096)
                if not chunk: break
                log.write(chunk[:max(0,65536-size)]); size+=len(chunk)
                if size>65536:
                    overflow.set(); process.kill()
    except BaseException as exc:
        pump_errors.append(type(exc).__name__)
        try: process.kill()
        except BaseException as stop_exc: cleanup_errors.append(type(stop_exc).__name__)
    finally: pipe.close()
def artifact_bytes():
    total=0; count=0
    for p in (PILOT_SESSION_ROOT/"run").rglob("*"):
        count+=1
        if count>4096: raise RuntimeError("worker file count limit")
        if p.is_file(): total+=p.stat().st_size
    for p in PILOT_SESSION_ROOT.iterdir():
        if p.is_file(): total+=p.stat().st_size
    return total
try:
    if not ENV_READY: raise RuntimeError("환경/source 사전검사 실패")
    command=[sys.executable,"-B","-m","rewind_m1.gpu_pilot","--profile","gpu","--job-id",JOB_ID,
             "--expected-revision",EXPECTED_REVISION,"--source-zip-sha256",EXPECTED_SOURCE_SHA256,
             "--output",str(PILOT_SESSION_ROOT/"run")]
    process=subprocess.Popen(command,cwd=PROJECT_ROOT,stdout=subprocess.PIPE,stderr=subprocess.PIPE,
        env={**os.environ,"PYTHONDONTWRITEBYTECODE":"1","PYTHONUNBUFFERED":"1","OMP_NUM_THREADS":"1",
             "MKL_NUM_THREADS":"1","CUBLAS_WORKSPACE_CONFIG":":4096:8"})
    supervision.update(started=True,worker_pid=process.pid)
    (PILOT_SESSION_ROOT/"supervision_started.json").write_text(json.dumps(supervision),encoding="utf8")
    for pipe,name in ((process.stdout,"stdout.log"),(process.stderr,"stderr.log")):
        thread=threading.Thread(target=pump,args=(pipe,PILOT_SESSION_ROOT/name),daemon=True)
        thread.start(); threads.append(thread)
    observed_rss=0
    while process.poll() is None:
        if time.monotonic()-started>=600: raise TimeoutError("worker 600s limit")
        try: rss=psutil.Process(process.pid).memory_info().rss
        except psutil.NoSuchProcess: break
        observed_rss=max(observed_rss,rss); disk=artifact_bytes()
        if rss>4*1024**3 or disk>128*1024**2: raise RuntimeError("supervisor resource limit")
        if overflow.is_set() or pump_errors: raise RuntimeError("worker log failure")
        time.sleep(.2)
    process.wait(timeout=5)
    supervision.update(returncode=process.returncode,observed_peak_worker_rss_bytes=observed_rss)
    result=json.loads((PILOT_SESSION_ROOT/"run/report.json").read_text(encoding="utf8"))
    supervision["status"]="passed" if process.returncode==0 and result.get("status")=="passed" else "failed"
except BaseException as exc:
    supervision.update(status="interrupted" if isinstance(exc,KeyboardInterrupt) else "timeout" if isinstance(exc,TimeoutError) else "failed",
                       error_type=type(exc).__name__,reason=str(exc)[:256])
finally:
    try:
        if process is not None and process.poll() is None:
            process.kill(); process.wait(timeout=5)
    except BaseException as exc: cleanup_errors.append(type(exc).__name__)
    for thread in threads: thread.join(timeout=5)
    alive=process is not None and process.poll() is None
    supervision.update(elapsed_seconds=time.monotonic()-started,child_alive=alive,
        log_overflow=overflow.is_set(),pump_errors=pump_errors,cleanup_errors=cleanup_errors,
        log_threads_alive=any(t.is_alive() for t in threads))
    if alive or pump_errors or cleanup_errors or supervision["log_threads_alive"]: supervision["status"]="failed"
    (PILOT_SESSION_ROOT/"supervision.json").write_text(json.dumps(supervision,ensure_ascii=False,indent=2),encoding="utf8")
print(supervision)
print("성공/실패/timeout 모두 회수 셀로 이동하세요. 실행 셀을 반복하지 않습니다.")
print("결과 폴더:",PILOT_SESSION_ROOT)


In [ ]:
# REPORT 변수/학습 worker 없이 디스크만 회수. 이 셀만 재실행할 수 있다.
import json, hashlib, zipfile, sys, os
from pathlib import Path
from google.colab import files
RECOVERY_ROOT = ""  # 필요하면 출력된 정확한 /content/rewind_gpu_pilot_<job> 경로를 입력
root=Path(RECOVERY_ROOT) if RECOVERY_ROOT else globals().get("PILOT_SESSION_ROOT")
if root is None:
    candidates=list(Path("/content").glob("rewind_gpu_pilot_*"))
    if len(candidates)!=1: raise RuntimeError("RECOVERY_ROOT를 테스트 세션이 알려준 정확한 결과 폴더로 설정하세요.")
    root=candidates[0]
root=Path(root)
if (root/"supervision_started.json").exists() and not (root/"supervision.json").exists():
    marker=json.loads((root/"supervision_started.json").read_text(encoding="utf8"))
    pid=marker.get("worker_pid")
    if pid:
        try: os.kill(pid,0)
        except ProcessLookupError: pass
        else: raise RuntimeError("worker가 살아 있습니다. 실행 셀 중단/종료 확인 후 회수하세요.")
if (root/"supervision.json").exists() and json.loads((root/"supervision.json").read_text()).get("child_alive"):
    raise RuntimeError("worker 종료 미확인: 재학습하지 말고 테스트 세션에 알리세요.")
# 성공한 source만 import한다. 회수 모듈은 stdlib 전용이며 torch/trainer를 import하지 않는다.
receipt=json.loads((root/"source_upload.json").read_text(encoding="utf8"))
if receipt.get("status")=="ready":
    projects=[p.parent.parent for p in (root/"source").rglob("rewind_m1/gpu_pilot_recovery.py")]
    if len(projects)!=1: raise RuntimeError("회수 모듈 경로 불일치")
    sys.path.insert(0,str(projects[0]))
    from rewind_m1.gpu_pilot_recovery import recover_bundle
    result=recover_bundle(root)
else:
    # source 업로드/검증 실패도 코드 실행 없이 작은 metadata ZIP으로 회수한다.
    path=root/"source_failure_result.zip"
    if not path.exists():
        entries=[p for p in root.iterdir() if p.is_file() and p.suffix in (".json",".log")]
        if len(entries)>16 or sum(p.stat().st_size for p in entries)>1024**2: raise RuntimeError("source 실패 회수 상한")
        with zipfile.ZipFile(path,"x",compression=zipfile.ZIP_DEFLATED) as z:
            for p in entries: z.write(p,p.name)
    if path.stat().st_size>2*1024**2: raise RuntimeError("source 실패 ZIP 상한")
    result={"path":str(path),"sha256":hashlib.sha256(path.read_bytes()).hexdigest(),"training_reexecuted":False}
print(result)
print("정확한 결과 ZIP:",result["path"])
files.download(result["path"])
